# SDXL-Turbo로 1초 만에 AI 이미지 생성하기 — Colab 무료 T4 GPU로 실습

위에서부터 차례로 실행해 보세요.

### 설치·환경 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q diffusers transformers accelerate

### SDXL Turbo fp16 로드

In [ ]:
import torch, time
from diffusers import AutoPipelineForText2Image
t=time.time()
pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16')
pipe.to('cuda'); torch.cuda.synchronize()
print('load s: %.1f'%(time.time()-t), 'VRAM GB: %.2f'%(torch.cuda.memory_allocated()/1e9))

### 1·2·4스텝 생성 시간 측정

In [ ]:
prompt='A cinematic shot of a baby racoon wearing an intricate italian priest robe.'
def gen(p, steps, seed=42, **kw):
    g=torch.Generator('cuda').manual_seed(seed)
    torch.cuda.synchronize(); t=time.time()
    im=p(prompt=prompt, num_inference_steps=steps, generator=g, **kw).images[0]
    torch.cuda.synchronize(); return im, time.time()-t
_,w=gen(pipe,1,guidance_scale=0.0,height=512,width=512); print('warmup s: %.2f'%w)
for s in (1,2,4):
    ts=[]
    for i in range(5):
        im,d=gen(pipe,s,guidance_scale=0.0,height=512,width=512); ts.append(d)
    im.save(f'turbo_{s}step.png'); print(s,'steps avg s: %.3f'%(sum(ts)/5))

### SDXL base 25스텝 비교

In [ ]:
base=AutoPipelineForText2Image.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')
for size in (512,1024):
    gen(base,5,height=size,width=size)
    ts=[]
    for i in range(3):
        im,d=gen(base,25,guidance_scale=7.5,height=size,width=size); ts.append(d)
    im.save(f'sdxl_25step_{size}.png'); print('SDXL 25 steps',size,'avg s: %.2f'%(sum(ts)/3))

### img2img

In [ ]:
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import load_image
i2i=AutoPipelineForImage2Image.from_pipe(pipe)
init=load_image('https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png').resize((512,512))
p2='cat wizard, gandalf, lord of the rings, detailed, fantasy, cute, adorable, Pixar, Disney, 8k'
for k in range(2):
    g=torch.Generator('cuda').manual_seed(1)
    torch.cuda.synchronize(); t=time.time()
    o=i2i(p2,image=init,num_inference_steps=2,strength=0.5,guidance_scale=0.0,generator=g).images[0]
    torch.cuda.synchronize(); print('img2img run',k,'%.3f'%(time.time()-t))
o.save('img2img_out.png')
del i2i, pipe
import gc; gc.collect(); torch.cuda.empty_cache()